In [1]:
# ---------------- Feature Selection ----------------

import pandas as pd
import numpy as np
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# ---------------- Step 1: Load Dataset ----------------

df = pd.read_csv(
    '../data/processed/Crimes_-_2001_to_Present_20260902.csv'
)

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (759290, 22)

Columns:
['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']


In [3]:
# ---------------- Step 2: Create Useful Time Features ----------------

df['Date'] = pd.to_datetime(
    df['Date'],
    format='%m/%d/%Y %I:%M:%S %p',
    errors='coerce'
)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Hour'] = df['Date'].dt.hour
df['DayOfWeek'] = df['Date'].dt.dayofweek

df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

print("Time features created successfully!")

Time features created successfully!


In [4]:
# ---------------- Step 3: Candidate Features ----------------

candidate_features = [
    'Year',
    'Month',
    'Hour',
    'DayOfWeek',
    'IsWeekend',
    'District',
    'Ward',
    'Community Area',
    'Beat',
    'Latitude',
    'Longitude',
    'Domestic',
    'Location Description'
]

available_features = [
    col for col in candidate_features
    if col in df.columns
]

print("Available candidate features:")
print(available_features)

Available candidate features:
['Year', 'Month', 'Hour', 'DayOfWeek', 'IsWeekend', 'District', 'Ward', 'Community Area', 'Beat', 'Latitude', 'Longitude', 'Domestic', 'Location Description']


In [5]:
# ---------------- Step 4: Remove Irrelevant / Identifier Features ----------------

irrelevant_features = [
    'ID',
    'Case Number',
    'Updated On',
    'Block'
]

features_to_remove = [
    col for col in irrelevant_features
    if col in df.columns
]

print("Features removed:")
print(features_to_remove)

Features removed:
['ID', 'Case Number', 'Updated On', 'Block']


In [6]:
# ---------------- Step 5: Select Important Features ----------------

selected_features = [
    col for col in available_features
    if col not in features_to_remove
]

selected_df = df[selected_features].copy()

print("Selected Features:")
print(selected_features)

print("\nSelected Dataset Shape:")
print(selected_df.shape)

Selected Features:
['Year', 'Month', 'Hour', 'DayOfWeek', 'IsWeekend', 'District', 'Ward', 'Community Area', 'Beat', 'Latitude', 'Longitude', 'Domestic', 'Location Description']

Selected Dataset Shape:
(759290, 13)


In [7]:
# ---------------- Step 6: Check Missing Values ----------------

missing_values = selected_df.isna().sum()

missing_values = missing_values[
    missing_values > 0
].sort_values(ascending=False)

print("Missing values in selected features:")
print(missing_values)

Missing values in selected features:
Latitude                5412
Longitude               5412
Location Description    3867
Community Area            35
Ward                       4
dtype: int64


In [8]:
# ---------------- Step 7: Handle Missing Values ----------------

numeric_columns = selected_df.select_dtypes(
    include=np.number
).columns

for col in numeric_columns:
    selected_df[col] = selected_df[col].fillna(
        selected_df[col].median()
    )

print("Missing values handled for numerical features.")

Missing values handled for numerical features.


In [9]:
# ---------------- Step 8: Check Feature Correlation ----------------

numeric_df = selected_df.select_dtypes(
    include=np.number
)

correlation_matrix = numeric_df.corr()

print(correlation_matrix.round(2))

                Year  Month  Hour  DayOfWeek  IsWeekend  District  Ward  \
Year            1.00  -0.01 -0.00      -0.00      -0.00     -0.01 -0.00   
Month          -0.01   1.00 -0.01       0.01       0.01      0.00  0.00   
Hour           -0.00  -0.01  1.00      -0.02      -0.04     -0.00  0.01   
DayOfWeek      -0.00   0.01 -0.02       1.00       0.79      0.01  0.00   
IsWeekend      -0.00   0.01 -0.04       0.79       1.00      0.01  0.00   
District       -0.01   0.00 -0.00       0.01       0.01      1.00  0.66   
Ward           -0.00   0.00  0.01       0.00       0.00      0.66  1.00   
Community Area  0.00  -0.01 -0.01      -0.01      -0.01     -0.48 -0.55   
Beat           -0.01   0.00 -0.00       0.01       0.01      1.00  0.66   
Latitude       -0.00   0.01  0.01       0.01       0.00      0.64  0.73   
Longitude       0.01   0.00  0.00      -0.00      -0.00     -0.54 -0.48   

                Community Area  Beat  Latitude  Longitude  
Year                      0.00 -0.01   

In [10]:
# ---------------- Step 9: Identify Highly Correlated Features ----------------

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

high_correlation = [
    column
    for column in upper_triangle.columns
    if any(abs(upper_triangle[column]) > 0.90)
]

print("Highly correlated features:")
print(high_correlation)

Highly correlated features:
['Beat']


In [11]:
# ---------------- Step 10: Final Selected Features ----------------

final_features = [
    'Year',
    'Month',
    'Hour',
    'DayOfWeek',
    'IsWeekend',
    'District',
    'Ward',
    'Community Area',
    'Beat',
    'Latitude',
    'Longitude',
    'Domestic'
]

final_features = [
    col for col in final_features
    if col in selected_df.columns
]

prepared_df = selected_df[final_features].copy()

print("Final Selected Features:")
for feature in final_features:
    print("-", feature)

print("\nPrepared Dataset Shape:", prepared_df.shape)

Final Selected Features:
- Year
- Month
- Hour
- DayOfWeek
- IsWeekend
- District
- Ward
- Community Area
- Beat
- Latitude
- Longitude
- Domestic

Prepared Dataset Shape: (759290, 12)


In [12]:
# ---------------- Step 11: Prepared Dataset Information ----------------

print("Prepared Dataset Information:")
prepared_df.info()

print("\nFirst 5 Rows:")
display(prepared_df.head())

Prepared Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 759290 entries, 0 to 759289
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Year            759290 non-null  int32  
 1   Month           759290 non-null  int32  
 2   Hour            759290 non-null  int32  
 3   DayOfWeek       759290 non-null  int32  
 4   IsWeekend       759290 non-null  int32  
 5   District        759290 non-null  int64  
 6   Ward            759290 non-null  float64
 7   Community Area  759290 non-null  float64
 8   Beat            759290 non-null  int64  
 9   Latitude        759290 non-null  float64
 10  Longitude       759290 non-null  float64
 11  Domestic        759290 non-null  bool   
dtypes: bool(1), float64(4), int32(5), int64(2)
memory usage: 50.0 MB

First 5 Rows:


,Year,Month,Hour,DayOfWeek,IsWeekend,District,Ward,Community Area,Beat,Latitude,Longitude,Domestic
0,2025,12,14,2,0,5,9.0,49.0,522,41.689820,-87.626713,False
1,2025,12,14,2,0,2,4.0,39.0,222,41.806890,-87.589834,False
2,2025,12,14,2,0,4,10.0,52.0,432,41.702049,-87.537704,True
3,2025,12,14,2,0,2,4.0,35.0,212,41.825124,-87.612294,False
4,2025,12,14,2,0,1,4.0,32.0,123,41.869759,-87.624121,False


In [13]:
# ---------------- Step 12: Save Prepared Dataset ----------------

output_path = (
    '../data/processed/chicago_crime_selected_features.csv'
)

prepared_df.to_csv(
    output_path,
    index=False
)

print("Prepared dataset saved successfully!")
print(output_path)

Prepared dataset saved successfully!
../data/processed/chicago_crime_selected_features.csv
